In [1]:
# ============================================================
# CELL 1 — Setup, GPU check, install PyTorch Geometric if needed
# Output: confirms GPU T4x2 + CUDA availability for GraphSAGE training
# ============================================================
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

try:
    import torch_geometric
    print("torch_geometric already installed:", torch_geometric.__version__)
except ImportError:
    import subprocess, sys
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "torch_geometric"], check=True)
    import torch_geometric
    print("torch_geometric installed:", torch_geometric.__version__)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", DEVICE)

CUDA available: True
GPU: Tesla T4
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 22.6 MB/s eta 0:00:00
torch_geometric installed: 2.8.0.post1
Using device: cuda


In [2]:
# ============================================================
# CELL 2 — Dataset paths (EDIT to match the dataset you published from Notebook A)
# ============================================================
EDGE_LIST_PATH     = "/kaggle/input/datasets/totallyapoorv/daemon-dino-opdata-v1/edge_list.csv"
NODE_FEATURES_PATH = "/kaggle/input/datasets/totallyapoorv/daemon-dino-opdata-v1/node_features.csv"
FULL_PANEL_PATH     = "/kaggle/input/datasets/totallyapoorv/daemon-dino-opdata-v1/full_panel.csv"
BORROWER_SPLIT_PATH = "/kaggle/input/datasets/totallyapoorv/daemon-dino-opdata-v1/borrower_split.csv"
import os
OUTPUT_DIR = "/kaggle/working/"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [3]:
# ============================================================
# CELL 3 — Build per-timestep PyG graph snapshots (dynamic features)
# FIXED: previously predicted "did this borrower EVER get stressed over
# 12 months" from purely STATIC attributes — none of which encode the
# one mechanism that actually drives stress (a neighbor's PRIOR state).
# That's why it landed at AUC 0.499. Now: one training example per
# borrower-TIMESTEP (matching Model 1/3 exactly), using dynamic
# own_lagged_risk / neighbor_stress_fraction / regional_shock_index from
# full_panel.csv. Filtered to own_lagged_risk == 0, same as Model 1/3,
# so all three models solve the same "genuine new onset" task and are
# directly comparable. Uses the SAME borrower-level split as Model 1/3
# (borrower_split.csv) — no borrower's months leak across train/test.
# Output: `snapshots` — list of (t, Data), each with train/test masks
#          restricted to eligible (own_lagged_risk==0) borrowers at that t
# ============================================================
import pandas as pd
import numpy as np
import torch
from torch_geometric.data import Data

edge_list = pd.read_csv(EDGE_LIST_PATH)
panel = pd.read_csv(FULL_PANEL_PATH)
split_df = pd.read_csv(BORROWER_SPLIT_PATH)

node_ids = sorted(panel["borrower_id"].unique())
id_to_idx = {bid: i for i, bid in enumerate(node_ids)}
n_nodes = len(node_ids)

src = edge_list["source"].map(id_to_idx)
dst = edge_list["target"].map(id_to_idx)
valid = src.notna() & dst.notna()
edge_index = torch.tensor(np.vstack([src[valid].values, dst[valid].values]), dtype=torch.long)
edge_index = torch.cat([edge_index, edge_index.flip(0)], dim=1)

train_borrowers = set(split_df[split_df["split"] == "train"]["borrower_id"])
test_borrowers  = set(split_df[split_df["split"] == "test"]["borrower_id"])

STATIC_COLS = ["loan_amount", "term_months", "current_savings"]
DYNAMIC_COLS = ["neighbor_stress_fraction", "regional_shock_index"]
FEATURE_COLS = DYNAMIC_COLS + STATIC_COLS + ["t"]  # own_lagged_risk excluded post-filter (constant 0)

panel_filtered = panel[panel["own_lagged_risk"] == 0].copy()
static_means = panel_filtered[STATIC_COLS].mean()

snapshots = []
for t, snap in panel_filtered.groupby("t"):
    snap = snap.set_index("borrower_id")
    x = np.zeros((n_nodes, len(FEATURE_COLS)), dtype=np.float32)
    y = np.zeros(n_nodes, dtype=np.float32)
    eligible = np.zeros(n_nodes, dtype=bool)
    train_mask = np.zeros(n_nodes, dtype=bool)
    test_mask = np.zeros(n_nodes, dtype=bool)

    for bid, idx in id_to_idx.items():
        if bid in snap.index:
            row = snap.loc[bid]
            x[idx] = [row.get(c, static_means.get(c, 0.0)) for c in FEATURE_COLS]
            y[idx] = row["is_stressed"]
            eligible[idx] = True
            if bid in train_borrowers:
                train_mask[idx] = True
            elif bid in test_borrowers:
                test_mask[idx] = True
        # nodes ineligible at this t (already stressed/defaulted) are
        # zero-featured and excluded from loss — this is fine because
        # neighbor_stress_fraction already encodes their contribution to
        # neighbors' features; the graph's edges mainly add value for
        # the optional multi-hop extension noted at the end.

    x_std = (x - x.mean(axis=0)) / (x.std(axis=0) + 1e-6)
    data_t = Data(
        x=torch.tensor(x_std, dtype=torch.float),
        edge_index=edge_index,
        y=torch.tensor(y, dtype=torch.float),
    )
    data_t.train_mask = torch.tensor(train_mask & eligible)
    data_t.test_mask = torch.tensor(test_mask & eligible)
    snapshots.append((int(t), data_t))

total_train = sum(s.train_mask.sum().item() for _, s in snapshots)
total_test  = sum(s.test_mask.sum().item() for _, s in snapshots)
print(f"OUTPUT: {len(snapshots)} timestep snapshots built — {n_nodes} nodes each, "
      f"{len(FEATURE_COLS)} features/node")
print(f"Total eligible train examples: {total_train}, test examples: {total_test}")
print(f"Positive rate across eligible rows: {panel_filtered['is_stressed'].mean():.2%}")

OUTPUT: 12 timestep snapshots built — 2000 nodes each, 6 features/node
Total eligible train examples: 16019, test examples: 3956
Positive rate across eligible rows: 6.57%


In [4]:
# ============================================================
# CELL 4 — GraphSAGE model definition (Model 2)
# Input dimension now matches Cell 3's dynamic feature set (6 features,
# was 4 static ones).
# ============================================================
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import SAGEConv

class GraphSAGE(nn.Module):
    def __init__(self, in_channels, hidden_channels=32):
        super().__init__()
        self.conv1 = SAGEConv(in_channels, hidden_channels)
        self.conv2 = SAGEConv(hidden_channels, hidden_channels)
        self.out = nn.Linear(hidden_channels, 1)

    def forward(self, x, edge_index):
        h = F.relu(self.conv1(x, edge_index))
        h = F.dropout(h, p=0.3, training=self.training)
        h = F.relu(self.conv2(h, edge_index))
        embeddings = h
        logits = self.out(h).squeeze(-1)
        return logits, embeddings

in_channels = snapshots[0][1].num_node_features
model = GraphSAGE(in_channels=in_channels).to(DEVICE)
snapshots = [(t, d.to(DEVICE)) for t, d in snapshots]

print(f"OUTPUT: GraphSAGE model created — {sum(p.numel() for p in model.parameters())} params, "
      f"in_channels={in_channels}, on {DEVICE}")

OUTPUT: GraphSAGE model created — 2529 params, in_channels=6, on cuda


In [5]:
# ============================================================
# CELL 5 — Train GraphSAGE across ALL timestep snapshots per epoch
# Each epoch: forward pass on every timestep's snapshot (same edge_index,
# different features/labels), sum the loss over each timestep's train
# mask, backprop once per epoch.
# ============================================================
from sklearn.metrics import roc_auc_score

optimizer = torch.optim.Adam(model.parameters(), lr=0.01, weight_decay=5e-4)
EPOCHS = 150

for epoch in range(1, EPOCHS + 1):
    model.train()
    optimizer.zero_grad()
    total_loss = 0.0
    n_terms = 0
    for t, snap in snapshots:
        if snap.train_mask.sum() == 0:
            continue
        logits, _ = model(snap.x, snap.edge_index)
        loss = F.binary_cross_entropy_with_logits(logits[snap.train_mask], snap.y[snap.train_mask])
        total_loss = total_loss + loss
        n_terms += 1
    if n_terms == 0:
        continue
    avg_loss = total_loss / n_terms
    avg_loss.backward()
    optimizer.step()

    if epoch % 10 == 0 or epoch == EPOCHS:
        model.eval()
        all_probs, all_labels = [], []
        with torch.no_grad():
            for t, snap in snapshots:
                if snap.test_mask.sum() == 0:
                    continue
                logits, _ = model(snap.x, snap.edge_index)
                probs = torch.sigmoid(logits).cpu().numpy()
                all_probs.append(probs[snap.test_mask.cpu().numpy()])
                all_labels.append(snap.y.cpu().numpy()[snap.test_mask.cpu().numpy()])
        all_probs = np.concatenate(all_probs)
        all_labels = np.concatenate(all_labels)
        val_auc = roc_auc_score(all_labels, all_probs) if len(set(all_labels)) > 1 else float("nan")
        print(f"Epoch {epoch:3d} | loss={avg_loss.item():.4f} | held_out_auc={val_auc:.3f}")

print("OUTPUT: training complete")

Epoch  10 | loss=0.2778 | held_out_auc=0.445
Epoch  20 | loss=0.2441 | held_out_auc=0.567
Epoch  30 | loss=0.2466 | held_out_auc=0.591
Epoch  40 | loss=0.2415 | held_out_auc=0.598
Epoch  50 | loss=0.2394 | held_out_auc=0.603
Epoch  60 | loss=0.2383 | held_out_auc=0.604
Epoch  70 | loss=0.2378 | held_out_auc=0.608
Epoch  80 | loss=0.2363 | held_out_auc=0.606
Epoch  90 | loss=0.2359 | held_out_auc=0.608
Epoch 100 | loss=0.2362 | held_out_auc=0.609
Epoch 110 | loss=0.2355 | held_out_auc=0.609
Epoch 120 | loss=0.2354 | held_out_auc=0.609
Epoch 130 | loss=0.2350 | held_out_auc=0.609
Epoch 140 | loss=0.2355 | held_out_auc=0.609
Epoch 150 | loss=0.2356 | held_out_auc=0.610
OUTPUT: training complete


In [6]:
# ============================================================
# CELL 6 — Save GraphSAGE weights + node embeddings (from the final
# available timestep snapshot, for the one-row-per-borrower handoff file)
# ============================================================
torch.save(model.state_dict(), OUTPUT_DIR + "graphsage_model.pt")

last_t, last_snap = snapshots[-1]
model.eval()
with torch.no_grad():
    logits, embeddings = model(last_snap.x, last_snap.edge_index)
    stress_prob = torch.sigmoid(logits).cpu().numpy()

emb_df = pd.DataFrame(embeddings.cpu().numpy(), columns=[f"emb_{i}" for i in range(embeddings.shape[1])])
emb_df.insert(0, "borrower_id", node_ids)
emb_df["predicted_stress_prob"] = stress_prob
emb_df["source_timestep"] = last_t
emb_df.to_csv(OUTPUT_DIR + "node_embeddings.csv", index=False)

print(f"OUTPUT: graphsage_model.pt saved ({sum(p.numel() for p in model.parameters())} params)")
print(f"OUTPUT: node_embeddings.csv saved — shape={emb_df.shape} (from t={last_t})")

OUTPUT: graphsage_model.pt saved (2529 params)
OUTPUT: node_embeddings.csv saved — shape=(2000, 35) (from t=11)


In [7]:
# ============================================================
# CELL 7 — Final held-out test-set evaluation (pooled across all timesteps)
# ============================================================
from sklearn.metrics import roc_auc_score, accuracy_score

model.eval()
all_probs, all_labels = [], []
with torch.no_grad():
    for t, snap in snapshots:
        if snap.test_mask.sum() == 0:
            continue
        logits, _ = model(snap.x, snap.edge_index)
        probs = torch.sigmoid(logits).cpu().numpy()
        all_probs.append(probs[snap.test_mask.cpu().numpy()])
        all_labels.append(snap.y.cpu().numpy()[snap.test_mask.cpu().numpy()])

all_probs = np.concatenate(all_probs)
all_labels = np.concatenate(all_labels)
all_preds = (all_probs > 0.5).astype(int)

print(f"Test AUC:      {roc_auc_score(all_labels, all_probs):.3f}")
print(f"Test Accuracy: {accuracy_score(all_labels, all_preds):.3f}")
print(f"Test set size: {len(all_labels)} borrower-timestep examples "
      f"({len(test_borrowers)} unique test borrowers)")

Test AUC:      0.610
Test Accuracy: 0.933
Test set size: 3956 borrower-timestep examples (400 unique test borrowers)
